In [ ]:

import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

import shap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print(" Imports loaded successfully!")

 Imports loaded successfully!


Why This Cell?
Purpose: Load all necessary libraries for the entire pipeline

Key Decisions:

warnings.filterwarnings("ignore") - Suppresses warnings to keep output clean

RANDOM_STATE = 42 - Ensures reproducible results across runs

Separate imports for sklearn, XGBoost, LightGBM, TensorFlow, and SHAP

Sets random seeds for NumPy and TensorFlow for reproducibility

In [8]:
from google.colab import drive
drive.mount('/content/drive')

# If using Google Drive, set paths here
# BASE_DIR = "/content/drive/MyDrive/ThyroidAI"
# If using local upload, use:
BASE_DIR = "/content"
DATA_PATH = os.path.join(BASE_DIR, "Thyroid_Diff.csv")
MODEL_DIR = os.path.join(BASE_DIR, "models")

os.makedirs(MODEL_DIR, exist_ok=True)
print(f"Data path: {DATA_PATH}")
print(f"Model directory: {MODEL_DIR}")

Mounted at /content/drive
Data path: /content/Thyroid_Diff.csv
Model directory: /content/models


Why This Cell?
Purpose: Set up file paths for data loading and model saving

Key Decisions:

Google Drive mounting allows persistent storage across sessions

os.makedirs(MODEL_DIR, exist_ok=True) creates directory if it doesn't exist

Paths are configurable - can switch between Drive and local storage

/content/ is Colab's default working directory

In [9]:
from google.colab import files
uploaded = files.upload()

# Check if file uploaded
if "Thyroid_Diff.csv" in uploaded:
    print(" File uploaded successfully!")
else:
    print(" Please upload Thyroid_Diff.csv")

 Please upload Thyroid_Diff.csv


Why This Cell?
Purpose: Upload the dataset from local machine to Colab

Key Decisions:

Uses Google Colab's built-in file upload widget

Verifies file was uploaded before proceeding

Alternative: Can download directly from UCI repository

In [11]:
TARGET_COL = "Recurred"


df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")


assert TARGET_COL in df.columns, f"Target column '{TARGET_COL}' not found"
assert df.shape[0] > 0, "Dataset is empty"

duplicate_count = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")
print(f"Target classes: {df[TARGET_COL].unique().tolist()}")
print(f"Class balance:\n{df[TARGET_COL].value_counts()}")

display(df.head())

Loaded 383 rows, 17 columns
Duplicate rows: 19
Target classes: ['No', 'Yes']
Class balance:
Recurred
No     275
Yes    108
Name: count, dtype: int64


,Age,Gender,Smoking,Hx Smoking,Hx Radiothreapy,Thyroid Function,Physical Examination,Adenopathy,Pathology,Focality,Risk,T,N,M,Stage,Response,Recurred
0,27,F,No,No,No,Euthyroid,Single nodular goiter-left,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Indeterminate,No
1,34,F,No,Yes,No,Euthyroid,Multinodular goiter,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
2,30,F,No,No,No,Euthyroid,Single nodular goiter-right,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
3,62,F,No,No,No,Euthyroid,Single nodular goiter-right,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
4,62,F,No,No,No,Euthyroid,Multinodular goiter,No,Micropapillary,Multi-Focal,Low,T1a,N0,M0,I,Excellent,No


Why This Cell?
Purpose: Load and validate the dataset

Key Decisions:

assert statements act as sanity checks - fail early if something's wrong

Check for duplicates to understand data quality

Display class balance - important for imbalanced classification

TARGET_COL = "Recurred" matches the dataset's target column name

display(df.head()) gives visual inspection of the data structure

## Missing value handling

In [12]:


missing_total = df.isnull().sum().sum()
print(f"Missing values found: {missing_total}")

if missing_total > 0:
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].fillna(df[c].mode()[0])
        else:
            df[c] = df[c].fillna(df[c].median())
    print("Missing values imputed (mode for categorical, median for numeric).")
else:
    print("No missing values — dataset is clean.")

Missing values found: 0
No missing values — dataset is clean.


Why This Cell?
Purpose: Handle missing values before modeling

Key Decisions:

Mode imputation for categorical variables - most frequent value

Median imputation for numeric variables - robust to outliers

Different strategies for different data types

Checks for missing values first to avoid unnecessary operations

## Encoding categorical features + scaling numeric features

In [13]:


feature_cols = [c for c in df.columns if c != TARGET_COL]
numeric_features = ["Age"]
categorical_features = [c for c in feature_cols if c not in numeric_features]

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(df[TARGET_COL])  # No=0, Yes=1
X = df[feature_cols].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)
preprocessor.fit(X_train)
X_train_t = preprocessor.transform(X_train)
X_test_t = preprocessor.transform(X_test)
print(f"Encoded feature dimensionality: {X_train_t.shape[1]}")

Train: (306, 16), Test: (77, 16)
Encoded feature dimensionality: 55


Why This Cell?
Purpose: Prepare features for machine learning models

Key Decisions:

LabelEncoder for target - converts "No"/"Yes" to 0/1

Stratified split - preserves class distribution in train/test

ColumnTransformer - applies different transformations to different columns

StandardScaler - standardizes Age to mean=0, std=1 (important for SVM, Logistic Regression)

OneHotEncoder - converts categorical variables to binary vectors

handle_unknown="ignore" - handles unseen categories in test data

80/20 train-test split with random_state for reproducibility

## Model training + evaluation

In [15]:


pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

sk_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=6, class_weight="balanced", random_state=RANDOM_STATE),
    "SVM": SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, eval_metric="logloss",
                              random_state=RANDOM_STATE, scale_pos_weight=pos_weight),
    "LightGBM": LGBMClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                random_state=RANDOM_STATE, class_weight="balanced", verbosity=-1),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []
fitted_models = {}
roc_curves = {}

for name, model in sk_models.items():
    cv_scores = cross_val_score(model, X_train_t, y_train, cv=skf, scoring="roc_auc")
    model.fit(X_train_t, y_train)
    y_pred = model.predict(X_test_t)
    y_proba = model.predict_proba(X_test_t)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_curves[name] = {"fpr": fpr.tolist(), "tpr": tpr.tolist()}

    metrics = {
        "model": name,
        "type": "ML",
        "cv_roc_auc_mean": float(cv_scores.mean()),
        "cv_roc_auc_std": float(cv_scores.std()),
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred)),
        "recall": float(recall_score(y_test, y_pred)),
        "f1": float(f1_score(y_test, y_pred)),
        "roc_auc": float(roc_auc_score(y_test, y_proba)),
    }
    results.append(metrics)
    fitted_models[name] = model
    print(f"{name:22s} | Acc {metrics['accuracy']:.4f} | F1 {metrics['f1']:.4f} | ROC-AUC {metrics['roc_auc']:.4f}")

Logistic Regression    | Acc 0.9610 | F1 0.9302 | ROC-AUC 0.9917
Random Forest          | Acc 0.9610 | F1 0.9268 | ROC-AUC 0.9950
SVM                    | Acc 0.9610 | F1 0.9302 | ROC-AUC 0.9843
KNN                    | Acc 0.9481 | F1 0.9048 | ROC-AUC 0.9851
Gradient Boosting      | Acc 0.9610 | F1 0.9268 | ROC-AUC 0.9851
XGBoost                | Acc 0.9610 | F1 0.9333 | ROC-AUC 0.9917
LightGBM               | Acc 0.9481 | F1 0.9130 | ROC-AUC 0.9934


Why This Cell?
Purpose: Train and evaluate multiple ML models

Key Decisions:

7 different models - diverse algorithms for comparison

Class weighting - handles imbalanced data (pos_weight for XGBoost, class_weight for others)

Cross-validation - 5-fold stratified CV gives more reliable performance estimates

ROC-AUC scoring - appropriate for imbalanced classification

Stores ROC curves - for later visualization

Tracks multiple metrics - accuracy, precision, recall, F1, ROC-AUC

Hyperparameter tuning - reasonable defaults for each model

## Training Deep Learning model: TensorFlow/Keras ANN

In [16]:
print("\nTraining Deep Learning model: TensorFlow/Keras ANN")
n_features = X_train_t.shape[1]
class_weight_dict = {0: 1.0, 1: float(pos_weight)}

ann = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
ann.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss="binary_crossentropy",
            metrics=["accuracy", keras.metrics.AUC(name="auc")])

early_stop = callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=25,
                                      restore_best_weights=True)

X_train_dense = X_train_t.toarray() if hasattr(X_train_t, "toarray") else X_train_t
X_test_dense = X_test_t.toarray() if hasattr(X_test_t, "toarray") else X_test_t

history = ann.fit(
    X_train_dense, y_train,
    validation_split=0.2,
    epochs=300,
    batch_size=16,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1,
)

y_proba_ann = ann.predict(X_test_dense, verbose=0).ravel()
y_pred_ann = (y_proba_ann >= 0.5).astype(int)
fpr, tpr, _ = roc_curve(y_test, y_proba_ann)
roc_curves["Neural Network (Keras ANN)"] = {"fpr": fpr.tolist(), "tpr": tpr.tolist()}

ann_metrics = {
    "model": "Neural Network (Keras ANN)",
    "type": "DL",
    "cv_roc_auc_mean": None,
    "cv_roc_auc_std": None,
    "accuracy": float(accuracy_score(y_test, y_pred_ann)),
    "precision": float(precision_score(y_test, y_pred_ann)),
    "recall": float(recall_score(y_test, y_pred_ann)),
    "f1": float(f1_score(y_test, y_pred_ann)),
    "roc_auc": float(roc_auc_score(y_test, y_proba_ann)),
}
results.append(ann_metrics)
print(f"{'Neural Network (ANN)':22s} | Acc {ann_metrics['accuracy']:.4f} | F1 {ann_metrics['f1']:.4f} | ROC-AUC {ann_metrics['roc_auc']:.4f} | epochs run: {len(history.history['loss'])}")


Training Deep Learning model: TensorFlow/Keras ANN
Epoch 1/300
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.7541 - auc: 0.6931 - loss: 0.9653 - val_accuracy: 0.8548 - val_auc: 0.9287 - val_loss: 0.5434
Epoch 2/300
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8770 - auc: 0.9220 - loss: 0.7601 - val_accuracy: 0.8710 - val_auc: 0.9443 - val_loss: 0.4018
Epoch 3/300
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9057 - auc: 0.9482 - loss: 0.5409 - val_accuracy: 0.8548 - val_auc: 0.9524 - val_loss: 0.3475
Epoch 4/300
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9057 - auc: 0.9686 - loss: 0.3820 - val_accuracy: 0.8548 - val_auc: 0.9606 - val_loss: 0.3456
Epoch 5/300
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9057 - auc: 0.9782 - loss: 0.2967 - val_accuracy: 0.8710 - val_auc: 0.9688 - val_loss: 0.2940
Epoch 6/300
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9344 - auc: 0.9808 - loss: 0.2670 - val_accuracy: 0.8871 - val_auc: 0.9762 - val_loss:

Why This Cell?
Purpose: Train a deep neural network for comparison

Key Decisions:

Architecture: 128 → 64 → 32 → 1 (decreasing layers)

Dropout - 30%, 20% to prevent overfitting

Early stopping - prevents overfitting, saves time

Class weights - handles imbalanced data

Adam optimizer - adaptive learning rate

Sigmoid output - binary classification

Validation split - 20% for monitoring

AUC metric - tracks area under ROC curve

Runs up to 300 epochs - stops early if no improvement

In [17]:
results_df = pd.DataFrame(results).sort_values(
    by=["roc_auc", "f1", "recall"], ascending=False
).reset_index(drop=True)
results_df = results_df.where(pd.notnull(results_df), None)

print("\n" + "=" * 70)
print("MODEL COMPARISON (sorted by ROC-AUC, then F1, then Recall)")
print("=" * 70)
display(results_df)

best_row = results_df.iloc[0]
best_name = best_row["model"]
print(f"\n>>> SELECTED BEST MODEL: {best_name}")

is_ann_best = best_name == "Neural Network (Keras ANN)"


MODEL COMPARISON (sorted by ROC-AUC, then F1, then Recall)


,model,type,cv_roc_auc_mean,cv_roc_auc_std,accuracy,precision,recall,f1,roc_auc
0,Neural Network (Keras ANN),DL,NaN,NaN,0.961039,0.952381,0.909091,0.930233,0.995041
1,Random Forest,ML,0.990909,0.008084,0.961039,1.000000,0.863636,0.926829,0.995041
2,LightGBM,ML,0.982635,0.011827,0.948052,0.875000,0.954545,0.913043,0.993388
3,XGBoost,ML,0.983422,0.017607,0.961039,0.913043,0.954545,0.933333,0.991736
4,Logistic Regression,ML,0.982635,0.019201,0.961039,0.952381,0.909091,0.930233,0.991736
5,Gradient Boosting,ML,0.984789,0.012359,0.961039,1.000000,0.863636,0.926829,0.985124
6,KNN,ML,0.953877,0.034390,0.948052,0.950000,0.863636,0.904762,0.985124
7,SVM,ML,0.982903,0.020969,0.961039,0.952381,0.909091,0.930233,0.984298



>>> SELECTED BEST MODEL: Neural Network (Keras ANN)


Why This Cell?
Purpose: Select the best model based on multiple criteria

Key Decisions:

Priority order: ROC-AUC → F1 → Recall

ROC-AUC is preferred for imbalanced data

F1 balances precision and recall

Recall handles the minority class (recurrence)

DataFrame sorted and displayed for comparison

Stored as results_df for later use

## Model saving

In [19]:


joblib.dump(preprocessor, os.path.join(MODEL_DIR, "preprocessing.pkl"))
joblib.dump(target_encoder, os.path.join(MODEL_DIR, "target_encoder.pkl"))
results_df.to_csv(os.path.join(MODEL_DIR, "model_comparison.csv"), index=False)

with open(os.path.join(MODEL_DIR, "roc_curves.json"), "w") as f:
    json.dump(roc_curves, f)

if is_ann_best:
    ann.save(os.path.join(MODEL_DIR, "best_model_ann.keras"))
    joblib.dump(None, os.path.join(MODEL_DIR, "best_model.pkl"))
else:
    joblib.dump(fitted_models[best_name], os.path.join(MODEL_DIR, "best_model.pkl"))

# Confusion matrix for best model
if is_ann_best:
    y_pred_best = y_pred_ann
else:
    y_pred_best = fitted_models[best_name].predict(X_test_t)
cm = confusion_matrix(y_test, y_pred_best)

categorical_options = {c: sorted(df[c].astype(str).unique().tolist()) for c in categorical_features}

metadata = {
    "best_model_name": best_name,
    "best_model_type": best_row["type"],
    "is_keras_model": bool(is_ann_best),
    "feature_cols": feature_cols,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "categorical_options": categorical_options,
    "target_classes": target_encoder.classes_.tolist(),
    "confusion_matrix": cm.tolist(),
    "dataset_info": {
        "n_samples": int(df.shape[0]),
        "n_features": len(feature_cols),
        "class_balance": df[TARGET_COL].value_counts().to_dict(),
    },
}
with open(os.path.join(MODEL_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2, default=str)

metrics_out = {
    "best_model": best_name,
    "selection_priority": ["roc_auc", "f1", "recall"],
    "all_models": results_df.to_dict(orient="records"),
    "best_model_metrics": best_row.to_dict(),
}
with open(os.path.join(MODEL_DIR, "metrics.json"), "w") as f:
    json.dump(metrics_out, f, indent=2, default=str)

print(f" All artifacts saved to {MODEL_DIR}")

 All artifacts saved to /content/models


Why This Cell?
Purpose: Save everything needed for deployment

Key Decisions:

joblib - efficient serialization for sklearn objects

Keras format - saved separately for neural networks

JSON files - human-readable format for metadata

Complete metadata - feature names, options, class balance

Model type detection - handles both sklearn and Keras models

Confusion matrix - stored for performance visualization

Categorical options - precomputed for frontend forms

In [ ]:
print("\nBuilding SHAP background dataset...")
background_idx = np.random.RandomState(RANDOM_STATE).choice(
    X_train_t.shape[0], size=min(50, X_train_t.shape[0]), replace=False
)
X_background = X_train_dense[background_idx]
joblib.dump(X_background, os.path.join(MODEL_DIR, "shap_background.pkl"))

ohe = preprocessor.named_transformers_["cat"]
ohe_feature_names = ohe.get_feature_names_out(categorical_features).tolist()
all_encoded_feature_names = numeric_features + ohe_feature_names
with open(os.path.join(MODEL_DIR, "encoded_feature_names.json"), "w") as f:
    json.dump(all_encoded_feature_names, f)

print(" SHAP background dataset created")

Why This Cell?
Purpose: Prepare for model explainability with SHAP

Key Decisions:

Background dataset - 50 random samples (sufficient for SHAP)

Background used - for KernelExplainer or TreeExplainer

Feature names - stored for interpretability

Precomputed - speeds up future SHAP explanations

Random selection - ensures representative sample

In [ ]:
from google.colab import files
import shutil


shutil.make_archive('/content/models', 'zip', '/content/models')
files.download('/content/models.zip')
print(" Models downloaded as models.zip")

# viz part

In [22]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings("ignore")

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
sns.set_context("notebook", font_scale=1.2)

In [24]:
# Create output directory for plots
OUT_DIR = os.path.join(BASE_DIR, "plots")
os.makedirs(OUT_DIR, exist_ok=True)


print("EXPLORATORY DATA ANALYSIS")

print(f"Plots will be saved to: {OUT_DIR}")

EXPLORATORY DATA ANALYSIS
Plots will be saved to: /content/plots


##  Dataset Overview

In [25]:
print(f"Total samples: {df.shape[0]}")
print(f"Total features: {df.shape[1]}")
print(f"Target column: {TARGET_COL}")
print(f"Target classes: {df[TARGET_COL].unique().tolist()}")

Total samples: 383
Total features: 17
Target column: Recurred
Target classes: ['No', 'Yes']


In [28]:
# Basic statistics
print("\n BASIC STATISTICS")


print(df.describe(include='all').round(2))




 BASIC STATISTICS
           Age Gender Smoking Hx Smoking Hx Radiothreapy Thyroid Function  \
count   383.00    383     383        383             383              383   
unique     NaN      2       2          2               2                5   
top        NaN      F      No         No              No        Euthyroid   
freq       NaN    312     334        355             376              332   
mean     40.87    NaN     NaN        NaN             NaN              NaN   
std      15.13    NaN     NaN        NaN             NaN              NaN   
min      15.00    NaN     NaN        NaN             NaN              NaN   
25%      29.00    NaN     NaN        NaN             NaN              NaN   
50%      37.00    NaN     NaN        NaN             NaN              NaN   
75%      51.00    NaN     NaN        NaN             NaN              NaN   
max      82.00    NaN     NaN        NaN             NaN              NaN   

       Physical Examination Adenopathy  Pathology   Foca

In [27]:
# Data types
print("\n DATA TYPES")

print(df.dtypes)


 DATA TYPES
Age                      int64
Gender                  object
Smoking                 object
Hx Smoking              object
Hx Radiothreapy         object
Thyroid Function        object
Physical Examination    object
Adenopathy              object
Pathology               object
Focality                object
Risk                    object
T                       object
N                       object
M                       object
Stage                   object
Response                object
Recurred                object
dtype: object


##  Target Distribution Analysis

In [30]:


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
ax1 = axes[0]
counts = df[TARGET_COL].value_counts()
colors = ['#2ecc71', '#e74c3c']  # Green for No, Red for Yes
bars = ax1.bar(counts.index, counts.values, color=colors, edgecolor='black', linewidth=1.5)
ax1.set_title('Target Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Recurrence')
ax1.set_ylabel('Count')
ax1.grid(axis='y', alpha=0.3)

# Add percentage labels on bars
total = len(df)
for bar, count in zip(bars, counts.values):
    height = bar.get_height()
    percentage = (count/total)*100
    ax1.text(bar.get_x() + bar.get_width()/2., height + 5,
             f'{count}\n({percentage:.1f}%)',
             ha='center', va='bottom', fontweight='bold')

# Pie chart
ax2 = axes[1]
explode = (0.05, 0.1)  # Explode the "Yes" slice
wedges, texts, autotexts = ax2.pie(counts.values,
                                   labels=counts.index,
                                   autopct='%1.1f%%',
                                   colors=colors,
                                   explode=explode,
                                   shadow=True,
                                   startangle=90,
                                   textprops={'fontsize': 12, 'weight': 'bold'})
ax2.set_title('Target Distribution (Pie Chart)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/1_target_distribution.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print(f" Target distribution plot saved to {OUT_DIR}")
print(f"Class distribution: No={counts.get('No', 0)} ({counts.get('No', 0)/len(df)*100:.1f}%), Yes={counts.get('Yes', 0)} ({counts.get('Yes', 0)/len(df)*100:.1f}%)")

 Target distribution plot saved to /content/plots
Class distribution: No=275 (71.8%), Yes=108 (28.2%)


## Age Distribution Analysis

In [31]:


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram with KDE
ax1 = axes[0]
df['Age'].hist(bins=30, edgecolor='black', alpha=0.7, color='#3498db', ax=ax1)
ax1.axvline(df['Age'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["Age"].mean():.1f}')
ax1.axvline(df['Age'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df["Age"].median():.1f}')
ax1.set_title('Age Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Age')
ax1.set_ylabel('Frequency')
ax1.legend()
ax1.grid(alpha=0.3)

# Box plot by recurrence
ax2 = axes[1]
sns.boxplot(x=TARGET_COL, y='Age', data=df, palette=['#2ecc71', '#e74c3c'], ax=ax2)
ax2.set_title('Age Distribution by Recurrence', fontsize=14, fontweight='bold')
ax2.set_xlabel('Recurrence')
ax2.set_ylabel('Age')
ax2.grid(alpha=0.3)

# Violin plot with swarm
ax3 = axes[2]
sns.violinplot(x=TARGET_COL, y='Age', data=df, palette=['#2ecc71', '#e74c3c'],
               split=True, inner='quartile', ax=ax3)
sns.swarmplot(x=TARGET_COL, y='Age', data=df, color='black', alpha=0.3, size=3, ax=ax3)
ax3.set_title('Age Violin Plot by Recurrence', fontsize=14, fontweight='bold')
ax3.set_xlabel('Recurrence')
ax3.set_ylabel('Age')
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/2_age_analysis.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print(" Age analysis plots saved")
print(f"Age statistics - Mean: {df['Age'].mean():.2f}, Median: {df['Age'].median():.2f}, Std: {df['Age'].std():.2f}")

 Age analysis plots saved
Age statistics - Mean: 40.87, Median: 37.00, Std: 15.13


## Categorical Features Analysis

In [32]:


categorical_cols = [c for c in df.columns if c not in ['Age', TARGET_COL] and df[c].dtype == 'object']

# Create subplots for categorical features
n_cols = 3
n_rows = (len(categorical_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5*n_rows))
axes = axes.flatten()

for idx, col in enumerate(categorical_cols):
    if idx < len(axes):
        ax = axes[idx]
        # Create cross-tabulation
        crosstab = pd.crosstab(df[col], df[TARGET_COL], normalize='index') * 100
        crosstab.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='black', linewidth=0.5)
        ax.set_title(f'{col}\n(Recurrence Rate by Category)', fontsize=11, fontweight='bold')
        ax.set_xlabel(col)
        ax.set_ylabel('Percentage (%)')
        ax.legend(['No Recurrence', 'Recurrence'], loc='upper right', fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Hide unused subplots
for idx in range(len(categorical_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/3_categorical_analysis.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print(f" Categorical features analysis saved ({len(categorical_cols)} features)")

 Categorical features analysis saved (15 features)


## Correlation Analysis

In [33]:


# Select only numeric columns for correlation
numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
numeric_cols_with_target = [c for c in numeric_cols if c != TARGET_COL]

# Also encode categorical variables for correlation
enc_df = df.copy()
for c in enc_df.columns:
    if not pd.api.types.is_numeric_dtype(enc_df[c]):
        enc_df[c] = LabelEncoder().fit_transform(enc_df[c].astype(str))

# Full correlation matrix with all encoded features
plt.figure(figsize=(14, 12))
correlation_matrix = enc_df.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix,
            mask=mask,
            annot=True,
            fmt=".2f",
            cmap="coolwarm",
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8},
            annot_kws={"size": 8})
plt.title('Correlation Heatmap (All Features)', fontsize=16, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/4_correlation_heatmap_full.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# Correlation with target
corr_with_target = correlation_matrix[TARGET_COL].sort_values(ascending=False)
plt.figure(figsize=(12, 6))
colors = ['green' if x > 0 else 'red' for x in corr_with_target.drop(TARGET_COL)]
corr_with_target.drop(TARGET_COL).plot(kind='bar', color=colors)
plt.title('Feature Correlation with Target (Recurrence)', fontsize=14, fontweight='bold')
plt.xlabel('Features')
plt.ylabel('Correlation Coefficient')
plt.xticks(rotation=45, ha='right')
plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.axhline(y=0.2, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
plt.axhline(y=-0.2, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/5_correlation_with_target.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print(f" Correlation analysis saved")
print(f"Top 5 correlated features with target:")
print(corr_with_target.drop(TARGET_COL).head(5))

 Correlation analysis saved
Top 5 correlated features with target:
Response    0.708957
N           0.632323
T           0.556201
Stage       0.449137
M           0.354360
Name: Recurred, dtype: float64


##  Feature Distribution by Target

In [34]:


# Select only numeric features (excluding Age which we already did)
numeric_features = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != TARGET_COL]

# Get top correlated numeric features
if len(numeric_features) > 0:
    corr_with_target = enc_df.corr()[TARGET_COL].drop(TARGET_COL)
    top_features = corr_with_target.abs().sort_values(ascending=False).head(6).index.tolist()

    # Filter to only numeric features
    top_numeric_features = [f for f in top_features if f in numeric_features]

    if len(top_numeric_features) > 0:
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        axes = axes.flatten()

        for idx, feature in enumerate(top_numeric_features[:6]):
            if idx < len(axes):
                ax = axes[idx]
                # Check if feature is numeric
                if pd.api.types.is_numeric_dtype(df[feature]):
                    # Box plot
                    data_no = df[df[TARGET_COL] == 'No'][feature].dropna()
                    data_yes = df[df[TARGET_COL] == 'Yes'][feature].dropna()

                    bp = ax.boxplot([data_no, data_yes],
                                   labels=['No', 'Yes'],
                                   patch_artist=True,
                                   medianprops=dict(color='black', linewidth=2))

                    # Color the boxes
                    bp['boxes'][0].set_facecolor('#2ecc71')
                    bp['boxes'][1].set_facecolor('#e74c3c')

                    ax.set_title(f'{feature}\n(by Recurrence)', fontsize=12, fontweight='bold')
                    ax.set_xlabel('Recurrence')
                    ax.set_ylabel('Value')
                    ax.grid(alpha=0.3)

        # Hide unused subplots
        for idx in range(len(top_numeric_features), len(axes)):
            axes[idx].set_visible(False)

        plt.tight_layout()
        plt.savefig(f"{OUT_DIR}/6_top_features_by_target.png", dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f" Top features distribution saved ({len(top_numeric_features)} features)")
    else:
        print(" No numeric features found for distribution plot")
else:
    print(" No numeric features found for distribution plot")

 No numeric features found for distribution plot


## Dimensionality Reduction (PCA & t-SNE)

In [35]:

# Prepare data for dimensionality reduction
X_encoded = enc_df.drop(columns=[TARGET_COL])
y_encoded = enc_df[TARGET_COL]

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_encoded)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA plot
ax1 = axes[0]
scatter1 = ax1.scatter(pca_result[:, 0], pca_result[:, 1],
                      c=y_encoded, cmap='RdYlGn', alpha=0.7,
                      edgecolors='black', linewidth=0.5, s=50)
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
ax1.set_title('PCA Visualization', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)
plt.colorbar(scatter1, ax=ax1, label='Recurrence (0=No, 1=Yes)')

# t-SNE (use subsample for speed if dataset is large)
try:
    if len(df) > 500:
        sample_idx = np.random.choice(len(df), 500, replace=False)
        tsne_data = X_encoded.iloc[sample_idx]
        tsne_labels = y_encoded.iloc[sample_idx]
    else:
        tsne_data = X_encoded
        tsne_labels = y_encoded

    tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30, n_iter=1000)
    tsne_result = tsne.fit_transform(tsne_data)

    ax2 = axes[1]
    scatter2 = ax2.scatter(tsne_result[:, 0], tsne_result[:, 1],
                          c=tsne_labels, cmap='RdYlGn', alpha=0.7,
                          edgecolors='black', linewidth=0.5, s=50)
    ax2.set_xlabel('t-SNE Component 1')
    ax2.set_ylabel('t-SNE Component 2')
    ax2.set_title('t-SNE Visualization', fontsize=14, fontweight='bold')
    ax2.grid(alpha=0.3)
    plt.colorbar(scatter2, ax=ax2, label='Recurrence (0=No, 1=Yes)')
except Exception as e:
    print(f" t-SNE plot error: {e}")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/7_dimensionality_reduction.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print(" Dimensionality reduction plots saved")

 Dimensionality reduction plots saved


## Pair Plot (Top Features)

In [36]:


# Select top numeric features for pairplot
numeric_features = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != TARGET_COL]
if len(numeric_features) >= 2:
    # Get top correlated features
    top_numeric = corr_with_target.abs().sort_values(ascending=False).head(5).index.tolist()
    top_numeric = [f for f in top_numeric if f in numeric_features]

    if len(top_numeric) >= 2:
        pairplot_data = df[top_numeric[:4] + [TARGET_COL]].copy()

        g = sns.pairplot(pairplot_data,
                        hue=TARGET_COL,
                        palette=['#2ecc71', '#e74c3c'],
                        diag_kind='kde',
                        markers=['o', 's'],
                        plot_kws={'alpha': 0.6, 'edgecolor': 'black', 'linewidth': 0.5})
        g.fig.suptitle('Pair Plot of Top Features', fontsize=16, fontweight='bold', y=1.02)

        plt.tight_layout()
        plt.savefig(f"{OUT_DIR}/8_pairplot_top_features.png", dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f" Pair plot saved with {len(top_numeric[:4])} features")
    else:
        print(" Not enough numeric features for pair plot")
else:
    print(" Not enough numeric features for pair plot")

 Not enough numeric features for pair plot


## Missing Value Analysis

In [37]:

missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0]

if len(missing_data) > 0:
    plt.figure(figsize=(10, 6))
    missing_data.plot(kind='bar', color='orange', edgecolor='black')
    plt.title('Missing Values by Feature', fontsize=14, fontweight='bold')
    plt.xlabel('Features')
    plt.ylabel('Number of Missing Values')
    plt.xticks(rotation=45, ha='right')
    for i, v in enumerate(missing_data):
        plt.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/9_missing_values.png", dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f" Missing values plot saved ({len(missing_data)} features with missing values)")
else:
    print(" No missing values found in dataset")

 No missing values found in dataset


##  Class Imbalance Analysis

In [38]:


fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Class distribution bar
ax = axes[0]
counts = df[TARGET_COL].value_counts()
colors_bar = ['#2ecc71' if label == 'No' else '#e74c3c' for label in counts.index]
bars = ax.bar(counts.index, counts.values, color=colors_bar, edgecolor='black', linewidth=1.5)
ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Recurrence Class')
ax.set_ylabel('Count')
ax.grid(axis='y', alpha=0.3)
for bar, count in zip(bars, counts.values):
    height = bar.get_height()
    percentage = (count/total)*100
    ax.text(bar.get_x() + bar.get_width()/2., height + 5,
            f'{count}\n({percentage:.1f}%)', ha='center', fontweight='bold')

# 2. Imbalance ratio
ax = axes[1]
no_count = counts.get('No', 0)
yes_count = counts.get('Yes', 0)
ratio = no_count / yes_count if yes_count > 0 else float('inf')

# Create a gauge-like visualization
imbalance_text = f'No/Yes Ratio: {ratio:.2f}\n\n'
if ratio > 5:
    imbalance_text += ' Severe Imbalance\n(Needs SMOTE/Random Sampling)'
elif ratio > 3:
    imbalance_text += ' High Imbalance\n(Consider Class Weights)'
elif ratio > 1.5:
    imbalance_text += ' Moderate Imbalance\n(Use Balanced Class Weight)'
else:
    imbalance_text += ' Balanced Dataset\n(No special handling needed)'

ax.text(0.5, 0.5, imbalance_text, horizontalalignment='center', verticalalignment='center',
        transform=ax.transAxes, fontsize=12, fontweight='bold')
ax.axis('off')
ax.set_title('Imbalance Analysis', fontsize=14, fontweight='bold')

# 3. Pie chart with annotation
ax = axes[2]
wedges, texts, autotexts = ax.pie(counts.values,
                                 labels=counts.index,
                                 autopct='%1.1f%%',
                                 colors=['#2ecc71', '#e74c3c'],
                                 explode=(0, 0.1),
                                 shadow=True,
                                 textprops={'fontsize': 12, 'weight': 'bold'})
ax.set_title('Class Proportions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/10_class_imbalance_analysis.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print(f" Class imbalance analysis saved")
print(f"Imbalance Ratio (No/Yes): {ratio:.2f}")

 Class imbalance analysis saved
Imbalance Ratio (No/Yes): 2.55


## EDA Summary Report

In [42]:


counts = df[TARGET_COL].value_counts()
no_count = counts.get('No', 0)
yes_count = counts.get('Yes', 0)
ratio = no_count / yes_count if yes_count > 0 else float('inf')

numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != TARGET_COL]
categorical_cols = [c for c in df.columns if c not in ['Age', TARGET_COL] and df[c].dtype == 'object']

print(f"""
 Dataset Overview:
- Total Samples: {df.shape[0]}
- Total Features: {df.shape[1]}
- Feature Types:
  * Numeric: {len(numeric_cols)} ({', '.join(numeric_cols)})
  * Categorical: {len(categorical_cols)}

 Target Distribution:
- No Recurrence: {no_count} ({no_count/len(df)*100:.1f}%)
- Recurrence: {yes_count} ({yes_count/len(df)*100:.1f}%)
- Imbalance Ratio: {ratio:.2f}

 Age Statistics:
- Mean: {df['Age'].mean():.2f}
- Median: {df['Age'].median():.2f}
- Std Dev: {df['Age'].std():.2f}
- Range: {df['Age'].min():.0f} - {df['Age'].max():.0f}

 Top Correlated Features:
{corr_with_target.head(5).to_string()}

 All visualization plots saved to: {OUT_DIR}
""")




 Dataset Overview:
- Total Samples: 383
- Total Features: 17
- Feature Types:
  * Numeric: 1 (Age)
  * Categorical: 15

 Target Distribution:
- No Recurrence: 275 (71.8%)
- Recurrence: 108 (28.2%)
- Imbalance Ratio: 2.55

 Age Statistics:
- Mean: 40.87
- Median: 37.00
- Std Dev: 15.13
- Range: 15 - 82

 Top Correlated Features:
Age                0.258897
Gender             0.328189
Smoking            0.333243
Hx Smoking         0.136073
Hx Radiothreapy    0.174407

 All visualization plots saved to: /content/plots

